# Alberi di decisione
l'algoritmo cerca di produrre un'albero decisionale.
Navigo l'albero, in riferimento ad un dato, con l'obiettivo di etichettarlo, in base a quale foglia raggiungo.

I nodi dell'albero costituiscono i test sulle feature, del tipo: $f_i < val$. L'esito di un test determina l'arco da prendere per navigare l'albero.

Si vuole creare questi alberi in modo da bilanciare il numero di confronti e l'incertezza (ossia, in base al numero di confronti potrei sbagliare!)

**Come si sceglie il test da effettuare su un nodo?** Deve essere il test che mi da maggior guadagno informativo, ossia che minimizza l'incertezza.

**Come si calcola l'incertezza?**: incertezza a sinistra, meno, incertezza a destra.

**Indice di Gini**: se e' basso allora i due sottoinsiemi sono sbilanciati!!!
$$
I_G (D_p) = \sum_c \frac{n_c}{n}(1- \frac{n_c}{n}) = 1 - \sum_c (\frac{n_c}{n})^2
$$
* $D_p$: insieme di esempi relativi al nodo $p$.
* $n = |D_p|$
* $n_c = \text{numero di esempi in } D_p \text{ appartenenti alla classe } c$.
* $n_c/n$: probabilita di assegnare un esempio alla classe $c$
* $n_c \to 0$: allora $I_G \to 0$
* se $I_G(D_p) =0$: se ho solo due classi ,vuol dire che una delle due  prevale sull'altra


ma quanto ci guadagno a passare da un nodo all'altro? 

**guadagno informativo**:
$$
IG(D_p) = I_G (D_p) - (\frac{|D_L|}{n}I_G(D_L) + \frac{n-|D_L|}{n}I_G(D_R))
$$
* ossia sommo l'indice del padre e poi peso i figli in base a quanti campioni hanno nel loro insieme

**concludendo**: devo scegliere in fase di addestramento feature e valori che mi dividono bene $D_R$ e $D_L$

**entropia**: $$I_H(D_p) = -\sum_c \frac{n_c}{n}log_2 \frac{n_c}{n}$$ e' simile alla funzione di Gini.

**impurita**: $I_G$ ed $I_H$ sono simili e dette funzioni di impurita

**foglia**: ottengo una foglia quando tutti gli oggetti nel nodo appartengono alla stessa clase. grazie al cazzo!

**generalizzazione ed overfitting**: chiaramente, un albero che divide perfettamente un'training set molto complesso potrebbe fare overfitting e performare di merda su altri dati! 

**soluzione**: limitare la profondita dell'albero con valori tra 3 e 4.

**iperparametri**: funzione impurita, altezza massima dell'albero, dimensione minima del dataset.


# Implementazione
**nodo**, dizionario contenente:
* `index`: indice della caratteristica usata.
* `value`: valore per il test
* `groups`: ci sono $D_L$ e $D_R$?
* `left, right`: i due figli

**nodo foglia**: un nodo dove il valore indica il nome della classe.

In [ ]:
import os
import pandas as pd
import numpy as np
from graphviz import Digraph # per il disegno degli alberi

In [ ]:
class DecisionTree(object):
    def __init__(self, max_depth=3, min_size=1):
        self.max_depth = max_depth
        self.min_size = min_size
        self.tree = None
        self._impurity_fun = self._gini

    def fit(self, X, y):
        """ Costruisce l'albero di decisione """
        y = np.array(y).reshape(-1, 1)  # una colonna x tutte le righe che servono
        
        '''
        dataset contiene sia X che y impilati verticalmente, questa è 
        la soluzione più conveniente per semplificare le operazioni
        di filtro delle righe che porterà alle suddivisioni del dataset che
        definiranno i nodi dell'albero
        '''
        dataset =  np.hstack((X, y)) # Concatenazione orizzontale
        self.tree = self._build_tree(dataset, 1)

    def _build_tree(self, dataset, depth):
        """ Costruisce l'albero a partire dai dati """
        root = self._get_best_split(dataset)
        self._split(root, depth)
        return root
    
    def _get_best_split(self, dataset):
        """ Trova la feature (colonna di dataset) sulla quale esiste un valore  che  
        Massimizza il guadagno informativo su tutte le possibili suddivisioni ottenibili usando
        tutte le possibili caratteristiche.

        Quindi per ogni caratteristica index e per ogni esempio row, si divide il dataset
        in base al test x[index] < row[index] e se ne calcola  il guadagno informativo.
        Si sceglie indx e row[index] in modo da massimizzare questo valore 
        """
        best_index, best_value, best_score, best_groups = None, None, float('-inf'), None
        for index in range(dataset.shape[1] - 1):  # Escludiamo la colonna target
            for row in dataset:
                groups = self._split_dataset(index, row[index], dataset)
                ig = self._info_gain(dataset, groups)
                if ig > best_score:
                    best_index, best_value, best_score, best_groups = index, row[index], ig, groups

        # ritorna un nodo
        return {'index': best_index, 'value': best_value, 'groups': best_groups}
    
    def _info_gain(self, dataset, groups):
        nl, nr = groups[0].shape[0], groups[1].shape[0]
        n = nl + nr
        ig = self._impurity_fun(dataset) - self._impurity_fun(groups[0])*nl/n - self._impurity_fun(groups[1])*nr/n
        return ig

    def _split_dataset(self, index, value, dataset):
        """ Divide il dataset in due gruppi in base al confronto della caratteristica
        index con value"""
        mask = dataset[:, index] < value
        left, right = dataset[mask], dataset[~mask]

        return left, right

    def _split(self, node, depth):
        """ Cresce l'albero ricorsivamente """
        left, right = node['groups']
        #del node['groups']
        
        # Se uno dei gruppi è vuoto, assegniamo una foglia
        if left.size == 0 or right.size == 0:
            node['left'] = node['right'] = self._create_leaf(np.vstack( (left, right) ))
            return

        # Fermiamo la crescita se abbiamo raggiunto la profondità massima
        if depth >= self.max_depth:
            node['left'], node['right'] = self._create_leaf(left), self._create_leaf(right)
            return

        # Se il gruppo sinistro è troppo piccolo, creiamo una foglia
        if len(left) <= self.min_size:
            node['left'] = self._create_leaf(left)
        else:
            node['left'] = self._get_best_split(left)
            self._split(node['left'], depth + 1)

        # Se il gruppo destro è troppo piccolo, creiamo una foglia
        if len(right) <= self.min_size:
            node['right'] = self._create_leaf(right)
        else:
            node['right'] = self._get_best_split(right)
            self._split(node['right'], depth + 1)

    def _create_leaf(self, group):
        """ Crea un nodo foglia con la classe più comune """
        values, counts = np.unique(group[:,-1], return_counts=True)
        return values[np.argmax(counts)]

    def _gini(self, dataset):
        # ricaviamo le etichette dall'ultima colonna
        labs, occur = np.unique(dataset[:,-1], return_counts=True)
        score = 0
        for i, _ in enumerate(labs):
            proportion = occur[i] / dataset.shape[0]
            score += proportion ** 2
        return 1-score

    def _predict_example(self, node, row):
        if row[node['index']] < node['value']:
            if self._is_leaf(node['left']):
                return node['left']
            else:
                return self._predict_example(node['left'], row)
                
        else:
            if self._is_leaf(node['right']):
                return node['right']
            else:
                return self._predict_example(node['right'], row)
            
    def _is_leaf(self, node):
        return not isinstance(node, dict)    
            
    def predict(self, row):
        """ Predice la classe di una singola riga """
        # innesca la ricerca a partire dalla radice dell'albero
        return self._predict_example(self.tree, row)

    def predict_batch(self, X):
        """ Predice su un intero dataset """
        return [self.predict(row) for row in X]
    
    def draw_tree(self):
        self.the_tree = Digraph()
    
        def add_nodes_edges(node, parent_id=None, edge_lab = 'SI'):
            if node is None:
                return
    
            # Se foglia (intero)
            if self._is_leaf(node):
                node_id = str(id(node))
                self.the_tree.node(node_id, str(node))
                if parent_id:
                    self.the_tree.edge(parent_id, node_id, edge_lab)
                return
    
            # Nodo interno
            node_id = str(id(node))
            label = f"f_{str(node.get('index',''))} < {str(node.get('value', ''))}" 
            self.the_tree.node(node_id, label)
    
            if parent_id:
                self.the_tree.edge(parent_id, node_id, edge_lab)
    
            add_nodes_edges(node.get('left'), node_id, 'SI')
            add_nodes_edges(node.get('right'), node_id, 'NO')
    
        add_nodes_edges(self.tree)

    def show_tree(self):
        return self.the_tree